# 🧠 PSC AI Trainer — LoRA Fine-tuning روی Kaggle (Llama 3.1 8B)
### نسخه بهینه‌سازی‌شده برای Kaggle P100/T4 (16GB VRAM)

---

## 📌 پیش‌نیازها و تنظیمات اولیه Kaggle

قبل از اجرا، موارد زیر را در پنل Kaggle انجام دهید:

### ۱. اضافه کردن Dataset
در پنل سمت راست Notebook → **Add Data** → **Upload Dataset**:
- فایل `PSC_AI_TRAINER_v3.jsonl` را آپلود کنید
- نام Dataset را `psc-training-data` بگذارید
- مسیر دسترسی: `/kaggle/input/psc-training-data/PSC_AI_TRAINER_v3.jsonl`

### ۲. انتخاب Accelerator
**Settings** → **Accelerator** → **GPU P100** (توصیه‌شده) یا T4 x2

### ۳. فعال‌سازی اینترنت
**Settings** → **Internet** → **On** (برای دانلود مدل از HuggingFace)

### ۴. Secret برای HuggingFace (اختیاری)
اگر مدل پایه نیاز به توکن دارد:  
**Add-ons** → **Secrets** → کلید `HF_TOKEN` را اضافه کنید

---

## 🗂️ ساختار خروجی‌ها

تمام خروجی‌ها در `/kaggle/working/` ذخیره می‌شوند (۲۰ GB سقف):

```
/kaggle/working/
├── psc-lora-final/          ← LoRA adapter نهایی
│   ├── adapter_config.json
│   ├── adapter_model.safetensors
│   └── psc_training_meta.json
├── checkpoints/             ← checkpoint هر ۵۰ گام
│   ├── checkpoint-step-50/
│   ├── checkpoint-step-100/
│   └── best_checkpoint/     ← بهترین مدل (کمترین eval loss)
├── psc-merged/              ← مدل ادغام‌شده (اگر فضا کافی باشد)
├── psc-q4_k_m.gguf          ← فایل GGUF نهایی
└── training_log.json        ← لاگ کامل آموزش
```

> **نکته Kaggle:** فایل‌های `/kaggle/working/` بعد از پایان session قابل دانلود هستند.  
> حداکثر فضا: ۲۰ GB — فایل GGUF حدود ۴.۵ GB خواهد بود.

---

## ⚡ مقایسه Colab vs Kaggle

| ویژگی | Colab T4 | Kaggle P100 | Kaggle T4×2 |
|-------|----------|-------------|-------------|
| VRAM | 15 GB | **16 GB** | 2×15 GB |
| RAM | 12 GB | **29 GB** | 29 GB |
| مدت Session | 12 ساعت | **12 ساعت** | 9 ساعت |
| هفتگی رایگان | محدود | **30 ساعت** | 30 ساعت |
| ذخیره‌سازی | Drive | **Working Dir** | Working Dir |

---


## ⬛ سلول ۱ — نصب کتابخانه‌ها

> **زمان تخمینی:** ۳–۵ دقیقه  
> فقط یک بار اجرا کنید. Kaggle نیازی به restart ندارد.

In [ ]:
import subprocess, sys

# ── تشخیص محیط ─────────────────────────────────────────────
import os
IS_KAGGLE = os.path.exists('/kaggle/working')
WORKING_DIR = '/kaggle/working' if IS_KAGGLE else '/content'
print(f"{'🟦 Kaggle' if IS_KAGGLE else '🟨 Colab'} محیط شناسایی شد")
print(f"Working directory: {WORKING_DIR}")

# ── بررسی GPU ───────────────────────────────────────────────
import subprocess
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                           '--format=csv,noheader'], capture_output=True, text=True)
print(f"\nGPU: {gpu_info.stdout.strip()}")

# ── نصب Unsloth (نسخه بهینه Kaggle) ────────────────────────
# Kaggle از CUDA 12.1 استفاده می‌کند
print("\n📦 نصب Unsloth...")
!pip install unsloth[cu121-torch230] -q 2>/dev/null || \
 pip install unsloth -q

# ── نصب وابستگی‌ها ───────────────────────────────────────────
print("📦 نصب وابستگی‌های اصلی...")
!pip install transformers>=4.40.0 accelerate peft bitsandbytes datasets trl -q

# ── llama.cpp برای تبدیل GGUF ─────────────────────────────
print("📦 کلون llama.cpp برای تبدیل GGUF...")
!git clone https://github.com/ggerganov/llama.cpp --depth=1 -q 2>/dev/null
!cd llama.cpp && make -j4 -s 2>/dev/null
!pip install -r llama.cpp/requirements.txt -q 2>/dev/null

print("\n✅ همه کتابخانه‌ها نصب شدند.")


## ⬛ سلول ۲ — پیکربندی کامل
> **تنها بخشی که نیاز به ویرایش دارد.**  
> مسیر فایل JSONL و پارامترهای آموزش را اینجا تنظیم کنید.

In [ ]:
import os, json, torch
from datetime import datetime

# ══════════════════════════════════════════════════════════
# ⚙️  بخش ۱: مسیرها
# ══════════════════════════════════════════════════════════

# مسیر فایل آموزشی (پس از آپلود Dataset در Kaggle)
# اگر نام Dataset را تغییر دادید، این مسیر را هم تغییر دهید
JSONL_PATH = "/kaggle/input/psc-training-data/PSC_AI_TRAINER_v3.jsonl"

# اگر فایل در جای دیگری است، مسیر واقعی را وارد کنید:
# JSONL_PATH = "/kaggle/input/YOUR_DATASET_NAME/PSC_AI_TRAINER_v3.jsonl"

# نام این run (برای سازماندهی خروجی‌ها)
RUN_NAME = f"psc-llama31-8b-{datetime.now().strftime('%Y%m%d-%H%M')}"

# ══════════════════════════════════════════════════════════
# ⚙️  بخش ۲: مسیرهای خروجی (خودکار در /kaggle/working/)
# ══════════════════════════════════════════════════════════
OUT           = WORKING_DIR
LORA_PATH     = f"{OUT}/psc-lora-final"
CKPT_PATH     = f"{OUT}/checkpoints"
MERGED_PATH   = f"{OUT}/psc-merged"
GGUF_PATH     = f"{OUT}/psc-q4_k_m.gguf"
LOG_PATH      = f"{OUT}/training_log.json"

for p in [LORA_PATH, CKPT_PATH, MERGED_PATH]:
    os.makedirs(p, exist_ok=True)

# ══════════════════════════════════════════════════════════
# ⚙️  بخش ۳: پارامترهای مدل
# ══════════════════════════════════════════════════════════
MODEL_CONFIG = {
    # مدل پایه — Unsloth نسخه کوانتیزه‌شده (سریع‌تر و سبک‌تر)
    "base_model"    : "unsloth/Meta-Llama-3.1-8B-Instruct",
    # طول توالی — با توجه به داده‌های PSC که برخی خروجی‌ها تا ۱۱k کاراکتر دارند
    # p90 ≈ 1761 توکن → 2048 پوشش مناسب است
    "max_seq_length": 2048,
    # بارگذاری در ۴ بیت برای صرفه‌جویی VRAM
    "load_in_4bit"  : True,
}

# ══════════════════════════════════════════════════════════
# ⚙️  بخش ۴: پارامترهای LoRA
# ══════════════════════════════════════════════════════════
LORA_CONFIG = {
    # r=16: تعادل خوب بین قدرت یادگیری و اندازه adapter
    # برای داده‌های تخصصی PSC (267 رکورد) مناسب است
    "r"            : 16,
    # alpha = 2×r: قرارداد استاندارد
    "lora_alpha"   : 32,
    "lora_dropout" : 0.05,
    # target_modules: تمام لایه‌های attention و FFN
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
}

# ══════════════════════════════════════════════════════════
# ⚙️  بخش ۵: پارامترهای آموزش
# ══════════════════════════════════════════════════════════
TRAIN_CONFIG = {
    # batch_size=2 + grad_accum=8 → effective batch=16
    # برای P100 با 16GB VRAM مناسب است
    "per_device_train_batch_size" : 2,
    "gradient_accumulation_steps": 8,
    # 3 epoch برای 267 رکورد کافی است (از overfitting جلوگیری می‌کند)
    "num_train_epochs"    : 3,
    # learning_rate استاندارد برای LoRA
    "learning_rate"       : 2e-4,
    # warmup برای جلوگیری از پریدن اولیه loss
    "warmup_ratio"        : 0.1,
    # ذخیره checkpoint هر 50 گام
    "save_steps"          : 50,
    "eval_steps"          : 50,
    "logging_steps"       : 10,
    # نگه داشتن فقط ۳ checkpoint آخر (صرفه‌جویی فضا در Kaggle)
    "save_total_limit"    : 3,
    # cosine: بهترین scheduler برای fine-tuning
    "lr_scheduler_type"   : "cosine",
    "weight_decay"        : 0.01,
    "max_grad_norm"       : 1.0,
    # packing=False چون برخی رکوردها طولانی هستند
    "packing"             : False,
    # نسبت داده‌های eval (10%)
    "eval_split"          : 0.10,
    "seed"                : 42,
}

# ══════════════════════════════════════════════════════════
# ⚙️  بخش ۶: تنظیمات GGUF
# ══════════════════════════════════════════════════════════
GGUF_CONFIG = {
    # q4_k_m: تعادل حجم/کیفیت — حدود 4.5 GB
    # گزینه‌های دیگر: q8_0 (8GB, کیفیت بالاتر) | q4_0 (4GB, سریع‌تر)
    "quantization": "q4_k_m",
}

# ── خلاصه پیکربندی ──────────────────────────────────────
print(f"{'='*55}")
print(f"⚙️  پیکربندی PSC AI Trainer")
print(f"{'='*55}")
print(f"  Run name     : {RUN_NAME}")
print(f"  JSONL path   : {JSONL_PATH}")
print(f"  LoRA output  : {LORA_PATH}")
print(f"  Checkpoints  : {CKPT_PATH}")
print(f"  GGUF output  : {GGUF_PATH}")
print(f"  Model        : {MODEL_CONFIG['base_model']}")
print(f"  Seq length   : {MODEL_CONFIG['max_seq_length']}")
print(f"  LoRA r       : {LORA_CONFIG['r']} | alpha: {LORA_CONFIG['lora_alpha']}")
print(f"  Batch (eff.) : {TRAIN_CONFIG['per_device_train_batch_size']} × {TRAIN_CONFIG['gradient_accumulation_steps']} = {TRAIN_CONFIG['per_device_train_batch_size']*TRAIN_CONFIG['gradient_accumulation_steps']}")
print(f"  Epochs       : {TRAIN_CONFIG['num_train_epochs']}")
print(f"  GGUF quant   : {GGUF_CONFIG['quantization']}")
print(f"{'='*55}")
print(f"  فضای آزاد /kaggle/working: ", end='')
usage = os.statvfs('/kaggle/working')
free_gb = (usage.f_bavail * usage.f_frsize) / (1024**3)
print(f"{free_gb:.1f} GB")


## ⬛ سلول ۳ — اعتبارسنجی و آنالیز داده‌های PSC
> بررسی کامل کیفیت فایل JSONL قبل از شروع آموزش.  
> **اگر فایل پیدا نشد:** از بخش پیکربندی `JSONL_PATH` را اصلاح کنید.

In [ ]:
from collections import Counter

def validate_and_analyze(path):
    """اعتبارسنجی کامل فایل JSONL آموزشی PSC"""

    print(f"📂 بررسی: {path}")

    # ── بررسی وجود فایل ─────────────────────────────────
    if not os.path.exists(path):
        print(f"\n❌ فایل پیدا نشد!")
        print(f"   مسیرهای موجود در /kaggle/input:")
        for root, dirs, files in os.walk('/kaggle/input'):
            for fname in files:
                if fname.endswith('.jsonl'):
                    print(f"   ✓ {os.path.join(root, fname)}")
        return None

    file_size = os.path.getsize(path) / 1024
    print(f"   حجم فایل: {file_size:.1f} KB")

    # ── خواندن و parse رکوردها ───────────────────────────
    records, errors, warnings = [], [], []
    with open(path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                output = rec.get('output', '').strip()
                if not output:
                    errors.append(f"خط {i}: output خالی")
                    continue
                if len(output) < 30:
                    warnings.append(f"خط {i} ({rec.get('id','?')}): output بسیار کوتاه ({len(output)} کاراکتر)")
                records.append(rec)
            except json.JSONDecodeError as e:
                errors.append(f"خط {i}: JSON نامعتبر — {e}")

    # ── گزارش کلی ───────────────────────────────────────
    print(f"\n📊 نتایج اعتبارسنجی:")
    print(f"   {'✅'} رکوردهای معتبر   : {len(records)}")
    print(f"   {'❌'} خطاهای جدی       : {len(errors)}")
    print(f"   {'⚠️'} هشدارها           : {len(warnings)}")

    if errors:
        print("\n❌ خطاها (۵ نمونه اول):")
        for e in errors[:5]: print(f"   {e}")

    # ── آمار انواع و سختی ───────────────────────────────
    types = Counter(r.get('type', 'N/A') for r in records)
    difficulties = Counter(r.get('difficulty_level', 'N/A') for r in records)

    print(f"\n📈 توزیع انواع رکورد:")
    type_icons = {
        'theoretical_foundation': '📚',
        'clinical_application'  : '🏥',
        'mechanism'             : '⚙️',
        'neural_mapping'        : '🧠',
        'intervention'          : '💊',
        'liminal_state'         : '🌊',
        'critique'              : '🔍',
        'general'               : '📝',
        'N/A'                   : '❓',
    }
    for t, c in types.most_common():
        icon = type_icons.get(t, '•')
        bar = '█' * c
        print(f"   {icon} {t:30s}: {c:3d}  {bar}")

    print(f"\n🎯 توزیع سختی:")
    diff_icons = {'advanced':'🔴','intermediate':'🟡','beginner':'🟢','N/A':'⚪'}
    for d, c in difficulties.most_common():
        print(f"   {diff_icons.get(d,'•')} {d:15s}: {c}")

    # ── آمار طول خروجی (مهم برای max_seq_length) ────────
    out_lens = sorted(len(r.get('output','')) for r in records)
    n = len(out_lens)
    token_est = lambda chars: chars // 3  # تخمین توکن برای متن فارسی (۳ کاراکتر/توکن)

    print(f"\n📏 آمار طول output (کاراکتر / توکن تخمینی):")
    for pct, label in [(10,'p10'),(25,'p25'),(50,'median'),(75,'p75'),(90,'p90'),(95,'p95')]:
        val = out_lens[int(n * pct/100)]
        print(f"   {label:8s}: {val:6,} کاراکتر ≈ {token_est(val):4,} توکن")
    print(f"   {'max':8s}: {out_lens[-1]:6,} کاراکتر ≈ {token_est(out_lens[-1]):4,} توکن")

    # توصیه max_seq_length
    p90_tokens = token_est(out_lens[int(n*0.9)])
    recommended = 1024 if p90_tokens < 512 else (2048 if p90_tokens < 1500 else 4096)
    print(f"\n💡 توصیه: max_seq_length = {recommended} (پوشش p90 داده‌ها)")
    if MODEL_CONFIG['max_seq_length'] < p90_tokens:
        print(f"   ⚠️  تنظیم فعلی ({MODEL_CONFIG['max_seq_length']}) کمتر از p90 است — کوتاه‌شدن داده‌ها رخ می‌دهد")
    else:
        print(f"   ✅ تنظیم فعلی ({MODEL_CONFIG['max_seq_length']}) مناسب است")

    # ── بررسی فرمت رکوردها ──────────────────────────────
    has_both = sum(1 for r in records if r.get('instruction','').strip() and r.get('input','').strip())
    only_input = sum(1 for r in records if not r.get('instruction','').strip() and r.get('input','').strip())
    print(f"\n🗂️  فرمت رکوردها:")
    print(f"   instruction + input  : {has_both}")
    print(f"   فقط input            : {only_input}")
    print(f"   فقط instruction      : {len(records) - has_both - only_input}")

    return records

records = validate_and_analyze(JSONL_PATH)
assert records is not None, "❌ فایل JSONL پیدا نشد یا خطا دارد — JSONL_PATH را اصلاح کنید"
print(f"\n✅ {len(records)} رکورد معتبر آماده آموزش است.")


## ⬛ سلول ۴ — آماده‌سازی Dataset با فرمت Llama 3.1
> تبدیل رکوردهای PSC به فرمت chat-template مناسب برای Llama 3.1 Instruct.  
> **سیستم پرامپت** شامل توضیح نقش PSC expert و راهنمای پاسخ دو زبانه است.

In [ ]:
from datasets import Dataset

# ── سیستم پرامپت PSC ────────────────────────────────────
PSC_SYSTEM_PROMPT = """You are a PSC (Psychological Semiotic Continuum) expert AI assistant.

The PSC model describes psychological functioning across three hierarchical levels:
• Level 1 — Survival/Reactive: fight-flight, amygdala-driven, automatic responses
• Level 2 — Adaptive/Social: learning, attachment, emotional regulation
• Level 3 — Metacognitive/Ethical: self-reflection, values, meaning-making

Key PSC concepts: Liminal States (transitions between levels), Neural Correlates,
Stress-Regression (downward shifts under stress), Clinical Applications.

Always respond in the SAME LANGUAGE as the user's question (Persian/Farsi or English).
Provide evidence-based, detailed, and structured answers."""

def build_prompt(record, tokenizer=None):
    """
    ساخت prompt با فرمت Llama 3.1 Instruct (ChatML)

    ساختار:
    <|begin_of_text|>
    <|start_header_id|>system<|end_header_id|>
    {system_prompt}<|eot_id|>
    <|start_header_id|>user<|end_header_id|>
    {user_content}<|eot_id|>
    <|start_header_id|>assistant<|end_header_id|>
    {response}<|eot_id|>
    """
    instruction = record.get('instruction', '').strip()
    context     = record.get('input', '').strip()
    response    = record.get('output', '').strip()

    # ترکیب instruction و input
    if instruction and context:
        # هر دو وجود دارند — instruction به عنوان راهنما
        user_content = f"{instruction}\n\n{context}"
    elif context:
        # فقط input (۲۴۹ رکورد از ۲۶۷)
        user_content = context
    else:
        user_content = instruction

    return {
        "text": (
            "<|begin_of_text|>"
            "<|start_header_id|>system<|end_header_id|>\n"
            f"{PSC_SYSTEM_PROMPT}"
            "<|eot_id|>"
            "<|start_header_id|>user<|end_header_id|>\n"
            f"{user_content}"
            "<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n"
            f"{response}"
            "<|eot_id|>"
        )
    }

# ── ساخت Dataset ─────────────────────────────────────────
formatted_records = [build_prompt(r) for r in records]
full_dataset = Dataset.from_list(formatted_records)

# ── تقسیم train/eval ─────────────────────────────────────
split = full_dataset.train_test_split(
    test_size=TRAIN_CONFIG['eval_split'],
    seed=TRAIN_CONFIG['seed']
)
train_dataset = split['train']
eval_dataset  = split['test']

# ── آمار Dataset ─────────────────────────────────────────
train_lens = [len(r['text']) for r in train_dataset]
print(f"✅ Dataset ساخته شد:")
print(f"   آموزش   : {len(train_dataset)} نمونه")
print(f"   ارزیابی : {len(eval_dataset)} نمونه")
print(f"   میانگین طول prompt: {int(sum(train_lens)/len(train_lens)):,} کاراکتر")
print(f"   بیشینه طول prompt : {max(train_lens):,} کاراکتر")

# ── نمایش یک نمونه ───────────────────────────────────────
print(f"\n📝 نمونه prompt (رکورد اول):"  )
print("-" * 55)
sample = train_dataset[0]['text']
print(sample[:600])
print("...(ادامه)...")
print("-" * 55)


## ⬛ سلول ۵ — بارگذاری مدل پایه با Unsloth
> **زمان تخمینی:** ۵–۱۰ دقیقه (دانلود + بارگذاری)  
> مدل از HuggingFace دانلود می‌شود — اینترنت Kaggle باید فعال باشد.

In [ ]:
import torch
from unsloth import FastLanguageModel

print(f"🔧 بارگذاری مدل پایه...")
print(f"   GPU  : {torch.cuda.get_device_name(0)}")
print(f"   VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"   RAM  : {os.popen('free -g | grep Mem').read().split()[1]} GB")
print(f"   مدل  : {MODEL_CONFIG['base_model']}")
print()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_CONFIG['base_model'],
    max_seq_length = MODEL_CONFIG['max_seq_length'],
    load_in_4bit   = MODEL_CONFIG['load_in_4bit'],
    # dtype=None → Unsloth خودکار float16 یا bfloat16 انتخاب می‌کند
    dtype          = None,
    device_map     = "auto",
    # cache_dir: ذخیره مدل در فضای موقت Kaggle
    cache_dir      = "/kaggle/working/model_cache",
)

print(f"\n✅ مدل بارگذاری شد.")
print(f"   VRAM مصرفی  : {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"   VRAM آزاد   : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.2f} GB")


## ⬛ سلول ۶ — اضافه کردن LoRA Adapter
> پارامترهای قابل آموزش: ~۸۴M از ۸B (حدود ۱٪) — بسیار کارآمد.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_CONFIG['r'],
    target_modules             = LORA_CONFIG['target_modules'],
    lora_alpha                 = LORA_CONFIG['lora_alpha'],
    lora_dropout               = LORA_CONFIG['lora_dropout'],
    bias                       = "none",
    # gradient_checkpointing: کاهش ۳۰٪ VRAM با هزینه ۲۰٪ سرعت
    use_gradient_checkpointing = "unsloth",
    random_state               = TRAIN_CONFIG['seed'],
    use_rslora                 = False,
    loftq_config               = None,
)

# ── آمار پارامترها ───────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params

print(f"✅ LoRA adapter اضافه شد:")
print(f"   پارامترهای کل         : {total_params/1e6:.1f}M")
print(f"   پارامترهای قابل آموزش : {trainable_params/1e6:.2f}M ({100*trainable_params/total_params:.2f}%)")
print(f"   پارامترهای منجمد      : {frozen_params/1e6:.1f}M")
print(f"   VRAM بعد از LoRA      : {torch.cuda.memory_allocated()/1e9:.2f} GB")


## ⬛ سلول ۷ — آموزش LoRA با Checkpoint خودکار
> **زمان تخمینی روی P100:** ~۴۵–۶۰ دقیقه (3 epoch روی ۲۴۰ نمونه)  
> 
> **Checkpoint‌ها در `/kaggle/working/checkpoints/` ذخیره می‌شوند.**  
> بهترین مدل (کمترین eval loss) در `best_checkpoint/` نگه داشته می‌شود.  
>
> **اگر session قطع شد:** سلول ۱۱ (بازیابی از checkpoint) را اجرا کنید.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, TrainerCallback
from unsloth import is_bfloat16_supported
import gc

# ══════════════════════════════════════════════════════════
# Callback: ذخیره checkpoint + لاگ + بهترین مدل
# ══════════════════════════════════════════════════════════
class KaggleCheckpointCallback(TrainerCallback):
    """
    ذخیره‌ساز هوشمند برای Kaggle:
    • هر save_steps گام، LoRA adapter ذخیره می‌شود
    • بهترین مدل (کمترین eval loss) جداگانه ذخیره می‌شود
    • لاگ کامل در training_log.json نوشته می‌شود
    • هشدار زمانی فضای دیسک کم باشد
    """

    def __init__(self, ckpt_path, log_path):
        self.ckpt_path  = ckpt_path
        self.log_path   = log_path
        self.history    = []
        self.best_eval  = float('inf')
        self.step_times = []
        self._last_time = datetime.now()

    def _disk_free_gb(self):
        s = os.statvfs('/kaggle/working')
        return (s.f_bavail * s.f_frsize) / (1024**3)

    def on_save(self, args, state, control, **kwargs):
        step = state.global_step
        free = self._disk_free_gb()

        # هشدار فضای کم
        if free < 3.0:
            print(f"\n⚠️  هشدار: فضای آزاد {free:.1f} GB — ذخیره checkpoint متوقف شد")
            return

        # ذخیره LoRA adapter
        ckpt_dir = os.path.join(self.ckpt_path, f"step-{step:05d}")
        os.makedirs(ckpt_dir, exist_ok=True)
        kwargs['model'].save_pretrained(ckpt_dir)
        kwargs['tokenizer'].save_pretrained(ckpt_dir)

        # متادیتای checkpoint
        last_eval = next((e.get('eval_loss') for e in reversed(self.history)
                         if 'eval_loss' in e), None)
        meta = {
            "step"       : step,
            "epoch"      : round(state.epoch, 3) if state.epoch else None,
            "eval_loss"  : last_eval,
            "disk_free_gb": round(free, 2),
            "saved_at"   : datetime.now().isoformat(),
        }
        with open(os.path.join(ckpt_dir, "meta.json"), 'w') as f:
            json.dump(meta, f, indent=2)

        # بهترین مدل (بر اساس eval loss)
        if last_eval and last_eval < self.best_eval:
            self.best_eval = last_eval
            best_dir = os.path.join(self.ckpt_path, "best_checkpoint")
            os.makedirs(best_dir, exist_ok=True)
            kwargs['model'].save_pretrained(best_dir)
            kwargs['tokenizer'].save_pretrained(best_dir)
            with open(os.path.join(best_dir, "meta.json"), 'w') as f:
                json.dump({**meta, "is_best": True, "best_eval_loss": self.best_eval}, f, indent=2)
            print(f"   🏆 بهترین مدل: step={step}, eval_loss={self.best_eval:.4f}, فضا={free:.1f}GB")
        else:
            print(f"   💾 Checkpoint step={step} | eval_loss={last_eval or 'N/A'} | فضا={free:.1f}GB")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        now = datetime.now()
        dt = (now - self._last_time).total_seconds()
        self._last_time = now

        entry = {"step": state.global_step, **logs,
                 "elapsed_sec": round(dt, 1),
                 "ts": now.isoformat()}
        self.history.append(entry)

        # ذخیره لاگ
        with open(self.log_path, 'w', encoding='utf-8') as f:
            json.dump(self.history, f, ensure_ascii=False, indent=2)

        # نمایش پیشرفت
        loss = logs.get('loss', '')
        eval_loss = logs.get('eval_loss', '')
        lr = logs.get('learning_rate', '')
        step = state.global_step
        total = state.max_steps
        pct = int(100 * step / total) if total else 0
        bar = '█' * (pct // 5) + '░' * (20 - pct // 5)
        print(f"   [{bar}] {pct:3d}% | step {step}/{total}",
              f"| loss={loss:.4f}" if isinstance(loss, float) else "",
              f"| eval={eval_loss:.4f}" if isinstance(eval_loss, float) else "",
              end='\r')

    def on_train_end(self, args, state, control, **kwargs):
        losses = [e['loss'] for e in self.history if 'loss' in e and isinstance(e['loss'], float)]
        eval_losses = [e['eval_loss'] for e in self.history if 'eval_loss' in e]
        summary = {
            "run_name"       : RUN_NAME,
            "total_steps"    : state.global_step,
            "total_epochs"   : round(state.epoch, 3) if state.epoch else None,
            "train_loss_init": round(losses[0], 4) if losses else None,
            "train_loss_final": round(losses[-1], 4) if losses else None,
            "loss_improvement_pct": round((1 - losses[-1]/losses[0])*100, 1) if len(losses)>1 else None,
            "best_eval_loss" : round(self.best_eval, 4) if self.best_eval != float('inf') else None,
            "completed_at"   : datetime.now().isoformat(),
            "num_train"      : len(train_dataset),
            "num_eval"       : len(eval_dataset),
            "config"         : {**MODEL_CONFIG, **LORA_CONFIG, **TRAIN_CONFIG},
        }
        sum_path = os.path.join(self.ckpt_path, "training_summary.json")
        with open(sum_path, 'w', encoding='utf-8') as f:
            json.dump(summary, f, ensure_ascii=False, indent=2)
        print(f"\n\n{'='*55}")
        print(f"✅ آموزش تمام شد!")
        print(f"   Loss اولیه  : {summary['train_loss_init']}")
        print(f"   Loss نهایی  : {summary['train_loss_final']}")
        print(f"   بهبود       : {summary['loss_improvement_pct']}%")
        print(f"   بهترین eval : {summary['best_eval_loss']}")
        print(f"   خلاصه در    : {sum_path}")

# ══════════════════════════════════════════════════════════
# TrainingArguments
# ══════════════════════════════════════════════════════════
training_args = TrainingArguments(
    output_dir                   = CKPT_PATH,
    per_device_train_batch_size  = TRAIN_CONFIG['per_device_train_batch_size'],
    per_device_eval_batch_size   = TRAIN_CONFIG['per_device_train_batch_size'],
    gradient_accumulation_steps  = TRAIN_CONFIG['gradient_accumulation_steps'],
    num_train_epochs             = TRAIN_CONFIG['num_train_epochs'],
    learning_rate                = TRAIN_CONFIG['learning_rate'],
    warmup_ratio                 = TRAIN_CONFIG['warmup_ratio'],
    lr_scheduler_type            = TRAIN_CONFIG['lr_scheduler_type'],
    weight_decay                 = TRAIN_CONFIG['weight_decay'],
    max_grad_norm                = TRAIN_CONFIG['max_grad_norm'],
    # precision: bfloat16 اگر P100 پشتیبانی کند، وگرنه float16
    fp16                         = not is_bfloat16_supported(),
    bf16                         = is_bfloat16_supported(),
    optim                        = "adamw_8bit",
    # ارزیابی و ذخیره بر اساس گام (نه epoch) — دقت بیشتر
    evaluation_strategy          = "steps",
    save_strategy                = "steps",
    eval_steps                   = TRAIN_CONFIG['eval_steps'],
    save_steps                   = TRAIN_CONFIG['save_steps'],
    save_total_limit             = TRAIN_CONFIG['save_total_limit'],
    load_best_model_at_end       = True,
    metric_for_best_model        = "eval_loss",
    greater_is_better            = False,
    logging_steps                = TRAIN_CONFIG['logging_steps'],
    report_to                    = "none",   # بدون WandB
    run_name                     = RUN_NAME,
    seed                         = TRAIN_CONFIG['seed'],
    # dataloader: چندنخی برای سرعت بیشتر در Kaggle
    dataloader_num_workers       = 2,
    dataloader_pin_memory        = True,
)

# ══════════════════════════════════════════════════════════
# SFTTrainer
# ══════════════════════════════════════════════════════════
callback = KaggleCheckpointCallback(CKPT_PATH, LOG_PATH)

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_dataset,
    eval_dataset       = eval_dataset,
    dataset_text_field = "text",
    max_seq_length     = MODEL_CONFIG['max_seq_length'],
    packing            = TRAIN_CONFIG['packing'],
    args               = training_args,
    callbacks          = [callback],
)

# پاک‌سازی حافظه قبل از شروع
torch.cuda.empty_cache()
gc.collect()

# ── شروع آموزش ──────────────────────────────────────────
print(f"🚀 شروع آموزش PSC LoRA")
print(f"{'='*55}")
print(f"  نمونه‌های آموزش   : {len(train_dataset)}")
print(f"  نمونه‌های ارزیابی : {len(eval_dataset)}")
print(f"  Effective batch   : {TRAIN_CONFIG['per_device_train_batch_size'] * TRAIN_CONFIG['gradient_accumulation_steps']}")
print(f"  Epochs            : {TRAIN_CONFIG['num_train_epochs']}")
print(f"  ذخیره checkpoint  : هر {TRAIN_CONFIG['save_steps']} گام")
print(f"{'='*55}\n")

trainer_stats = trainer.train()

print(f"\n⏱️  مدت آموزش: {trainer_stats.metrics.get('train_runtime', 0)/60:.1f} دقیقه")


## ⬛ سلول ۸ — ذخیره LoRA Adapter نهایی
> LoRA adapter نهایی به همراه متادیتای کامل ذخیره می‌شود.

In [ ]:
print(f"💾 ذخیره LoRA adapter نهایی...")

model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)

# ── متادیتای کامل ────────────────────────────────────────
final_meta = {
    "run_name"           : RUN_NAME,
    "base_model"         : MODEL_CONFIG['base_model'],
    "dataset"            : os.path.basename(JSONL_PATH),
    "num_train_samples"  : len(train_dataset),
    "num_eval_samples"   : len(eval_dataset),
    "train_loss_final"   : trainer_stats.metrics.get('train_loss'),
    "train_runtime_min"  : round(trainer_stats.metrics.get('train_runtime',0)/60, 2),
    "lora_r"             : LORA_CONFIG['r'],
    "lora_alpha"         : LORA_CONFIG['lora_alpha'],
    "max_seq_length"     : MODEL_CONFIG['max_seq_length'],
    "epochs"             : TRAIN_CONFIG['num_train_epochs'],
    "learning_rate"      : TRAIN_CONFIG['learning_rate'],
    "effective_batch"    : TRAIN_CONFIG['per_device_train_batch_size'] * TRAIN_CONFIG['gradient_accumulation_steps'],
    "saved_at"           : datetime.now().isoformat(),
    "lora_path"          : LORA_PATH,
    "checkpoint_path"    : CKPT_PATH,
    "gpu"                : torch.cuda.get_device_name(0),
}

meta_path = os.path.join(LORA_PATH, "psc_training_meta.json")
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(final_meta, f, ensure_ascii=False, indent=2)

# ── بررسی فایل‌های ذخیره‌شده ────────────────────────────
print(f"\n✅ LoRA adapter ذخیره شد: {LORA_PATH}")
print(f"\n   فایل‌های ذخیره‌شده:")
for fname in sorted(os.listdir(LORA_PATH)):
    fpath = os.path.join(LORA_PATH, fname)
    if os.path.isfile(fpath):
        fsize_mb = os.path.getsize(fpath) / (1024**2)
        print(f"   📄 {fname:<45s} {fsize_mb:>7.1f} MB")


## ⬛ سلول ۹ — تست مدل آموزش‌دیده
> بررسی کیفیت پاسخ‌ها قبل از ادغام و تبدیل به GGUF.

In [ ]:
FastLanguageModel.for_inference(model)

def ask_psc(question: str, max_new_tokens: int = 512) -> str:
    """ارسال سوال به مدل PSC و دریافت پاسخ"""
    prompt = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n"
        f"{PSC_SYSTEM_PROMPT}"
        "<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{question}"
        "<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens  = max_new_tokens,
            temperature     = 0.7,
            top_p           = 0.9,
            repetition_penalty = 1.1,
            do_sample       = True,
            pad_token_id    = tokenizer.eos_token_id,
        )
    resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return resp.strip()

# ── تست‌های سطح‌بندی‌شده ────────────────────────────────
tests = [
    # سطح مقدماتی
    ("مدل PSC چیست؟", "beginner"),
    # سطح متوسط — دو زبانه
    ("What is a Liminal State in the PSC model and how does stress affect it?", "intermediate"),
    # سطح پیشرفته — از داده‌های واقعی
    ("رابطه استرس و پسرفت سطح PSC را بر اساس یافته‌های Arnsten (2015) و Schüren et al. (2026) تبیین کنید.", "advanced"),
]

for i, (q, level) in enumerate(tests, 1):
    print(f"\n{'═'*60}")
    print(f"🧪 تست {i} [{level}]")
    print(f"❓ {q[:90]}{'...' if len(q)>90 else ''}")
    print(f"{'─'*60}")
    ans = ask_psc(q)
    print(ans[:700])
    if len(ans) > 700:
        print(f"...({len(ans)-700} کاراکتر دیگر)")
    print()

print("✅ تست‌ها کامل شدند.")


## ⬛ سلول ۱۰ — ادغام LoRA با مدل پایه
> **زمان تخمینی:** ۱۰–۱۵ دقیقه  
> **فضای مورد نیاز:** ~۱۶ GB (برای مدل merged)  
> 
> ⚠️ **هشدار Kaggle:** اگر فضای کافی نیست، مستقیم از سلول ۱۱ (تبدیل GGUF بدون merge) استفاده کنید.

In [ ]:
import gc

# بررسی فضای آزاد
free_gb = (os.statvfs('/kaggle/working').f_bavail *
           os.statvfs('/kaggle/working').f_frsize) / (1024**3)
print(f"💽 فضای آزاد: {free_gb:.1f} GB")

if free_gb < 16:
    print(f"⚠️  فضای کافی برای مدل merged وجود ندارد ({free_gb:.1f} GB < 16 GB)")
    print(f"   برای GGUF از روش مستقیم استفاده می‌شود (سلول بعدی)")
    MERGE_DONE = False
else:
    print(f"🔀 شروع ادغام LoRA...")
    torch.cuda.empty_cache()
    gc.collect()
    print(f"   VRAM قبل: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    merged_model = model.merge_and_unload()

    print(f"   VRAM بعد : {torch.cuda.memory_allocated()/1e9:.2f} GB")
    print(f"\n💾 ذخیره مدل merged (این ممکن است ۱۵+ دقیقه طول بکشد)...")

    merged_model.save_pretrained(MERGED_PATH, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_PATH)

    total_size = sum(
        os.path.getsize(os.path.join(MERGED_PATH, f))
        for f in os.listdir(MERGED_PATH)
        if os.path.isfile(os.path.join(MERGED_PATH, f))
    ) / (1024**3)

    print(f"\n✅ مدل merged ذخیره شد.")
    print(f"   حجم: {total_size:.2f} GB")
    print(f"   مسیر: {MERGED_PATH}")
    MERGE_DONE = True


## ⬛ سلول ۱۱ — تبدیل به GGUF
> سه روش تبدیل، به ترتیب اولویت:  
> 1. **Unsloth** (اگر مدل merged دارید)  
> 2. **Unsloth مستقیم** (بدون merge — سریع‌تر برای Kaggle)  
> 3. **llama.cpp** (fallback نهایی)

In [ ]:
GGUF_DONE = False

# ── روش ۱: Unsloth از مدل merged ─────────────────────────
if MERGE_DONE and not GGUF_DONE:
    print(f"🔄 روش ۱: تبدیل GGUF از مدل merged با Unsloth...")
    try:
        merged_model.save_pretrained_gguf(
            GGUF_PATH, tokenizer,
            quantization_method=GGUF_CONFIG['quantization']
        )
        GGUF_DONE = True
        print(f"✅ روش ۱ موفق بود!")
    except Exception as e:
        print(f"⚠️  روش ۱ ناموفق: {e}")

# ── روش ۲: Unsloth مستقیم از LoRA ────────────────────────
if not GGUF_DONE:
    print(f"🔄 روش ۲: تبدیل مستقیم LoRA → GGUF با Unsloth...")
    try:
        # فعال‌سازی حالت training برای save_pretrained_gguf
        FastLanguageModel.for_training(model)
        model.save_pretrained_gguf(
            GGUF_PATH, tokenizer,
            quantization_method=GGUF_CONFIG['quantization']
        )
        GGUF_DONE = True
        print(f"✅ روش ۲ موفق بود!")
    except Exception as e:
        print(f"⚠️  روش ۲ ناموفق: {e}")

# ── روش ۳: llama.cpp fallback ────────────────────────────
if not GGUF_DONE:
    print(f"🔄 روش ۳: تبدیل با llama.cpp...")

    if not MERGE_DONE:
        print(f"   ابتدا باید مدل merged وجود داشته باشد.")
        print(f"   سلول ۱۰ را با حافظه کافی اجرا کنید.")
    else:
        llama_cpp_convert = "llama.cpp/convert_hf_to_gguf.py"
        if not os.path.exists(llama_cpp_convert):
            print(f"   کلون llama.cpp...")
            !git clone https://github.com/ggerganov/llama.cpp --depth=1 -q
            !cd llama.cpp && make -j4 -s

        !python {llama_cpp_convert} \
            "{MERGED_PATH}" \
            --outfile "{GGUF_PATH}" \
            --outtype {GGUF_CONFIG['quantization']}

        if os.path.exists(GGUF_PATH):
            GGUF_DONE = True
            print(f"✅ روش ۳ موفق بود!")

# ── نتیجه نهایی ──────────────────────────────────────────
if GGUF_DONE and os.path.exists(GGUF_PATH):
    gguf_gb = os.path.getsize(GGUF_PATH) / (1024**3)
    print(f"\n{'='*55}")
    print(f"✅ فایل GGUF آماده است!")
    print(f"   حجم     : {gguf_gb:.2f} GB")
    print(f"   مسیر    : {GGUF_PATH}")
    print(f"   کمیت    : {GGUF_CONFIG['quantization']}")
    print(f"{'='*55}")
else:
    print(f"\n❌ تبدیل GGUF ناموفق بود.")
    print(f"   LoRA adapter در {LORA_PATH} در دسترس است.")
    print(f"   می‌توانید بعداً روی سرور خود تبدیل کنید.")


## ⬛ سلول ۱۲ — گزارش نهایی و دستورالعمل دانلود

In [ ]:
print("="*60)
print("📋 گزارش نهایی PSC AI Trainer")
print("="*60)
print(f"  Run     : {RUN_NAME}")
print(f"  زمان    : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print()

# ── فایل‌های خروجی ───────────────────────────────────────
print("📁 فایل‌های خروجی در /kaggle/working/:")
output_files = []
for root, dirs, files in os.walk(WORKING_DIR):
    # فقط پوشه‌های اصلی
    if root.count('/') > WORKING_DIR.count('/') + 1:
        continue
    for fname in files:
        fpath = os.path.join(root, fname)
        fsize = os.path.getsize(fpath)
        output_files.append((fpath, fsize))
        rel = fpath.replace(WORKING_DIR + '/', '')
        print(f"   📄 {rel:<50s} {fsize/(1024**2):>8.1f} MB")

total_out = sum(s for _, s in output_files) / (1024**3)
print(f"\n   مجموع: {total_out:.2f} GB")

# ── آمار آموزش ───────────────────────────────────────────
if os.path.exists(LOG_PATH):
    with open(LOG_PATH) as f:
        log = json.load(f)
    losses = [e['loss'] for e in log if isinstance(e.get('loss'), float)]
    evals  = [e['eval_loss'] for e in log if isinstance(e.get('eval_loss'), float)]
    print(f"\n📈 خلاصه آموزش:")
    print(f"   گام‌های کل      : {log[-1].get('step','N/A')}")
    if losses:
        print(f"   Train loss اول  : {losses[0]:.4f}")
        print(f"   Train loss آخر  : {losses[-1]:.4f}")
        print(f"   بهبود loss      : {(1-losses[-1]/losses[0])*100:.1f}%")
    if evals:
        print(f"   Eval loss بهترین: {min(evals):.4f}")

# ── راهنمای دانلود ────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"📥 دانلود فایل‌ها از Kaggle:")
print(f"   ۱. در پنل Notebook → Data → Output")
print(f"   ۲. یا از منوی سمت راست → Output Files")
print(f"   ۳. روی فایل GGUF کلیک کرده و Download کنید")

print(f"\n🚀 اجرا با Ollama:")
print(f"   # ساخت Modelfile")
print(f"   echo 'FROM ./psc-q4_k_m.gguf' > Modelfile")
print(f"   echo 'SYSTEM \"You are a PSC AI expert.\"' >> Modelfile")
print(f"   ollama create psc-ai -f Modelfile")
print(f"   ollama run psc-ai")

print(f"\n🚀 اجرا با llama.cpp:")
print(f"   ./llama-cli -m psc-q4_k_m.gguf --ctx-size 2048 -p 'سوال شما'")

print(f"\n{'='*60}")


---
## ⬛ سلول ۱۳ — بازیابی از Checkpoint (در صورت قطع Session)
> اگر Kaggle session قطع شد و می‌خواهید آموزش را ادامه دهید.  
> این سلول را فقط در صورت نیاز اجرا کنید.

In [ ]:
def resume_from_checkpoint():
    """
    بازیابی و ادامه آموزش از آخرین checkpoint ذخیره‌شده.

    مراحل:
    1. پیدا کردن آخرین checkpoint در /kaggle/working/checkpoints/
    2. بارگذاری مجدد مدل پایه
    3. بارگذاری LoRA از checkpoint
    4. ادامه آموزش با resume_from_checkpoint
    """
    import glob

    # پیدا کردن checkpoint‌ها
    ckpt_dirs = sorted(
        glob.glob(os.path.join(CKPT_PATH, "step-*")),
        key=lambda x: int(x.split("-")[-1])
    )

    if not ckpt_dirs:
        print(f"❌ هیچ checkpoint‌ای پیدا نشد در: {CKPT_PATH}")
        print(f"   آموزش را از ابتدا (سلول ۵) شروع کنید.")
        return None

    latest = ckpt_dirs[-1]
    step = int(latest.split("-")[-1])

    # خواندن متادیتا
    meta_file = os.path.join(latest, "meta.json")
    if os.path.exists(meta_file):
        with open(meta_file) as f:
            meta = json.load(f)
        print(f"✅ آخرین checkpoint:")
        print(f"   مسیر      : {latest}")
        print(f"   گام       : {meta.get('step')}")
        print(f"   Epoch     : {meta.get('epoch')}")
        print(f"   Eval loss : {meta.get('eval_loss', 'N/A')}")
        print(f"   ذخیره در  : {meta.get('saved_at')}")
    else:
        print(f"✅ Checkpoint یافت شد: step={step}")

    # بارگذاری مجدد
    print(f"\n🔄 بارگذاری مدل از checkpoint...")
    from unsloth import FastLanguageModel
    from peft import PeftModel

    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = MODEL_CONFIG['base_model'],
        max_seq_length = MODEL_CONFIG['max_seq_length'],
        load_in_4bit   = MODEL_CONFIG['load_in_4bit'],
        dtype          = None,
        device_map     = "auto",
        cache_dir      = "/kaggle/working/model_cache",
    )

    model = FastLanguageModel.get_peft_model(
        base_model,
        r              = LORA_CONFIG['r'],
        target_modules = LORA_CONFIG['target_modules'],
        lora_alpha     = LORA_CONFIG['lora_alpha'],
        lora_dropout   = LORA_CONFIG['lora_dropout'],
        bias           = "none",
        use_gradient_checkpointing = "unsloth",
        random_state   = TRAIN_CONFIG['seed'],
    )

    # بارگذاری وزن‌های checkpoint
    model.load_adapter(latest)
    print(f"✅ LoRA adapter از checkpoint بارگذاری شد.")
    print(f"\n💡 برای ادامه آموزش:")
    print(f"   trainer.train(resume_from_checkpoint='{latest}')")

    return model, tokenizer, latest

# اجرای بازیابی
# model, tokenizer, resume_ckpt = resume_from_checkpoint()
print("💡 این سلول را فقط در صورت قطع session اجرا کنید.")
print("   برای فعال‌سازی، کامنت سطر آخر را بردارید.")
